# Build RAG Index on Kaggle GPU

1. Upload `records.jsonl` as Kaggle dataset `guap-raw`
2. Enable GPU T4
3. Run All
4. Download `rag_index.tar.gz` from Output

In [ ]:
!pip install -q transformers==4.51.0 sentence-transformers==5.1.1 einops torch faiss-cpu numpy huggingface-hub bs4 pypdf python-docx pymupdf pytesseract flagembedding

In [ ]:
!rm -rf /tmp/llm-speaker-core
!GIT_LFS_SKIP_SMUDGE=1 GIT_TERMINAL_PROMPT=0 git clone https://github.com/chudinovAI/llm-speaker-core.git /tmp/llm-speaker-core

import sys
sys.path.insert(0, '/tmp/llm-speaker-core/src')
print('OK')

In [ ]:
from pathlib import Path

RAW_CANDIDATES = [
    Path('/kaggle/input/guap-raw/records.jsonl'),
    Path('/tmp/llm-speaker-core/data/raw/cloudflare/latest/records.jsonl'),
]
RAW_RECORDS = next((p for p in RAW_CANDIDATES if p.exists()), None)
assert RAW_RECORDS is not None, f'records.jsonl not found in {RAW_CANDIDATES}'
print(f'Using: {RAW_RECORDS}')

OUT_DIR = Path('/kaggle/working/rag_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'indexes/bm25').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'indexes/faiss').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'normalized').mkdir(parents=True, exist_ok=True)

In [ ]:
from llm_speaker_core.retrieval.build import build_hybrid_index

report = build_hybrid_index(
    raw_records=RAW_RECORDS,
    documents_out=OUT_DIR / 'normalized/documents.jsonl',
    chunks_out=OUT_DIR / 'normalized/chunks.jsonl',
    manifest_out=OUT_DIR / 'index_manifest.json',
    lexical_out=OUT_DIR / 'indexes/bm25/index.json',
    dense_out=OUT_DIR / 'indexes/faiss/index.json',
    embedding_model='ai-sage/Giga-Embeddings-instruct',
    reranker_model='BAAI/bge-reranker-v2-m3',
    manual_docs_dir=None,
)
print(report)

In [ ]:
import json

manifest = json.loads((OUT_DIR / 'index_manifest.json').read_text())
print(json.dumps(manifest, indent=2, ensure_ascii=False))
assert manifest['metadata']['dense_available'] == True, 'ERROR: Dense index NOT built!'
print('\n✅ Dense index built successfully')

In [ ]:
import tarfile

archive_path = '/kaggle/working/rag_index.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(OUT_DIR / 'index_manifest.json', arcname='data/index_manifest.json')
    tar.add(OUT_DIR / 'indexes/bm25/index.json', arcname='data/indexes/bm25/index.json')
    tar.add(OUT_DIR / 'indexes/faiss/index.json', arcname='data/indexes/faiss/index.json')
    tar.add(OUT_DIR / 'indexes/faiss/index.vectors.npy', arcname='data/indexes/faiss/index.vectors.npy')
    faiss_path = OUT_DIR / 'indexes/faiss/index.faiss'
    if faiss_path.exists():
        tar.add(faiss_path, arcname='data/indexes/faiss/index.faiss')
    tar.add(OUT_DIR / 'normalized/documents.jsonl', arcname='data/normalized/documents.jsonl')
    tar.add(OUT_DIR / 'normalized/chunks.jsonl', arcname='data/normalized/chunks.jsonl')

size_mb = Path(archive_path).stat().st_size / 1024 / 1024
print(f'\n📦 Archive: {archive_path} ({size_mb:.1f} MB)')
print('Download and run: cd llm-speaker-core && tar xzf ~/Downloads/rag_index.tar.gz')